In [1]:
!git clone https://github.com/swetha3456/TiSASRec.pytorch.git
%cd TiSASRec.pytorch

Cloning into 'TiSASRec.pytorch'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 110 (delta 66), reused 63 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 6.89 MiB | 17.37 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/kaggle/working/TiSASRec.pytorch


In [2]:
import json
import pandas as pd
from datetime import datetime
from tqdm import tqdm

def process_json_data(input_path, output_path):
    processed_rows = []
    
    print(f"Reading JSON from {input_path}...")
    with open(input_path, 'r', encoding='utf-8') as f:
        # Load the entire JSON object
        data = json.load(f)
        
    print(f"Processing {len(data)} users...")
    # Iterate through the list of data (with a progress bar)
    for entry in tqdm(data):
        uid = entry['user_id']
        
        # 1. Extract History (fast_memory_ids)
        history_ids = entry.get('fast_memory_ids', [])
        history_ts = entry.get('fast_id_timestamps', [])
        
        for item, ts in zip(history_ids, history_ts):
            dt_obj = datetime.strptime(ts, '%Y-%m-%d %H:%M:%S')
            processed_rows.append([uid, item, int(dt_obj.timestamp())])
            
        # 2. Extract the Target Item
        target_item = entry.get('target_item')
        target_ts = entry.get('target_timestamp')
        if target_item and target_ts:
            target_dt = datetime.strptime(target_ts, '%Y-%m-%d %H:%M:%S')
            processed_rows.append([uid, target_item, int(target_dt.timestamp())])

    # Create DataFrame
    df = pd.DataFrame(processed_rows, columns=['user_id', 'item_id', 'timestamp'])
    
    # Sort chronologically (Crucial for sequential models)
    df = df.sort_values(by=['user_id', 'timestamp'])

    # 3. Map IDs to 1-indexed integers
    print("Mapping IDs to integers...")
    user_map = {val: i + 1 for i, val in enumerate(df['user_id'].unique())}
    item_map = {val: i + 1 for i, val in enumerate(df['item_id'].unique())}
    
    df['user_id'] = df['user_id'].map(user_map)
    df['item_id'] = df['item_id'].map(item_map)

    # 4. Save as Tab-Separated (no header)
    df.to_csv(output_path, sep='\t', index=False, header=False)
    
    print(f"--- Done ---")
    print(f"Saved to: {output_path}")
    return item_map

# Run the function
item_mapping = process_json_data('/kaggle/input/datasets/swetha344/mind-sequential-dataset-2/processed_mind_memories.json', 'data/mind.txt')

Reading JSON from /kaggle/input/datasets/swetha344/mind-sequential-dataset-2/processed_mind_memories.json...
Processing 71616 users...


100%|██████████| 71616/71616 [00:14<00:00, 5091.36it/s]


Mapping IDs to integers...
--- Done ---
Saved to: data/mind.txt


In [3]:
def create_mind_category_mapping(news_path, item_map, json_output_path):
    """
    news_path: Path to news.tsv
    item_map: The item_map dict returned from the preprocessing function
    """
    print("Creating category mapping...")
    # News columns: NewsID, Category, SubCategory, Title, Abstract, URL...
    news_df = pd.read_csv(news_path, sep='\t', usecols=[0, 1], names=['item_id', 'category'])
    
    # Map Category strings to integers
    cat_list = news_df['category'].unique()
    cat_map = {cat: i + 1 for i, cat in enumerate(cat_list)}
    
    # Create the final dictionary: { "IntegerItemID": IntegerCategoryID }
    final_mapping = {}
    
    # We only care about items that survived the filtering in the main script
    for _, row in news_df.iterrows():
        raw_id = row['item_id']
        if raw_id in item_map:
            mapped_item_id = str(item_map[raw_id]) # JSON keys must be strings
            mapped_cat_id = cat_map[row['category']]
            final_mapping[mapped_item_id] = mapped_cat_id
            
    with open(json_output_path, 'w') as f:
        json.dump({"product_to_category": final_mapping}, f)
        
    print(f"Category mapping saved to: {json_output_path}")
    print(f"Total items in mapping: {len(final_mapping)}")

create_mind_category_mapping('/kaggle/input/datasets/xuntngtrn/mind-large-train/news.tsv', item_mapping, 'data/category_mapping.json')

Creating category mapping...
Category mapping saved to: data/category_mapping.json
Total items in mapping: 7471


In [4]:
!python main.py --dataset=mind --train_dir=default --device=cuda --lr=2e-5 --hidden_units=4 --dropout_rate=0.5 --num_epochs=100

Loading category mappings for hard negative sampling...
Preparing data...
Preparing done...
average sequence length: 249.84
Preparing relation matrix: 100%|███████████| 5960/5960 [00:08<00:00, 676.00it/s]
loss in epoch 1 iteration 0: 1.3814704418182373
loss in epoch 1 iteration 10: 1.3815770149230957
loss in epoch 1 iteration 20: 1.3816118240356445
loss in epoch 1 iteration 30: 1.381425380706787
loss in epoch 1 iteration 40: 1.3815772533416748
loss in epoch 2 iteration 0: 1.3812063932418823
loss in epoch 2 iteration 10: 1.3818715810775757
loss in epoch 2 iteration 20: 1.38194739818573
loss in epoch 2 iteration 30: 1.3806095123291016
loss in epoch 2 iteration 40: 1.3815478086471558
loss in epoch 3 iteration 0: 1.381174087524414
loss in epoch 3 iteration 10: 1.3809516429901123
loss in epoch 3 iteration 20: 1.3818608522415161
loss in epoch 3 iteration 30: 1.3814001083374023
loss in epoch 3 iteration 40: 1.3804203271865845
loss in epoch 4 iteration 0: 1.3805617094039917
loss in epoch 4 ite